<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Custom_OCR_Training_Detector_Recognizer/blob/main/PNID_OCR_Detection_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [TESTING] Text Detector Model Training

### Repo Clone & Installation

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import paddle

paddle.device.get_device()
paddle.set_device("gpu")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 325904, done.
remote: Counting objects: 100% (1651/1651), done.
remote: Compressing objects: 100% (344/344), done.
remote: Total 325904 (delta 1494), reused 1307 (delta 1307), pack-reused 324253 (from 3)
Receiving objects: 100% (325904/325904), 1.73 GiB | 29.47 MiB/s, done.
Resolving deltas: 100% (258161/258161), done.


In [ ]:
%cd PaddleOCR

!pip install -r requirements.txt

/content/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.2 MB/s eta 0:00:00


### Processing Data

In [14]:
# 下载示例数据集
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/data/ocr_det_dataset_examples.tar
!tar -xf ocr_det_dataset_examples.tar

--2026-05-01 10:05:01--  https://paddle-model-ecology.bj.bcebos.com/paddlex/data/ocr_det_dataset_examples.tar
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22773760 (22M) [application/octet-stream]
Saving to: ‘ocr_det_dataset_examples.tar’

ocr_det_dataset_exa 100%[===================>]  21.72M  5.65MB/s    in 24s     

2026-05-01 10:05:27 (929 KB/s) - ‘ocr_det_dataset_examples.tar’ saved [22773760/22773760]



In [ ]:
# 下载 PP-OCRv5_server_det 预训练模型
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams

--2026-03-07 10:38:05--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105414496 (101M) [application/octet-stream]
Saving to: ‘PP-OCRv5_server_det_pretrained.pdparams’

PP-OCRv5_server_det 100%[===================>] 100.53M  16.8MB/s    in 29s     

2026-03-07 10:38:36 (3.49 MB/s) - ‘PP-OCRv5_server_det_pretrained.pdparams’ saved [105414496/105414496]



### Training Model

In [ ]:
#单卡训练 (默认训练方式)
!python3 tools/train.py -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
    Train.dataset.data_dir=./ocr_det_dataset_examples \
    Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
    Eval.dataset.data_dir=./ocr_det_dataset_examples \
    Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'


#多卡训练，通过--gpus参数指定卡号
# python3 -m paddle.distributed.launch --gpus '0,1,2,3' tools/train.py \
#     -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
#     -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
#     Train.dataset.data_dir=./ocr_det_dataset_examples \
#     Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
#     Eval.dataset.data_dir=./ocr_det_dataset_examples \
#     Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/03/07 10:38:48] ppocr INFO: Architecture : 
[2026/03/07 10:38:48] ppocr INFO:     Backbone : 
[2026/03/07 10:38:48] ppocr INFO:         det : True
[2026/03/07 10:38:48] ppocr INFO:         name : PPHGNetV2_B4
[2026/03/07 10:38:48] ppocr INFO:     Head : 
[2026/03/07 10:38:48] ppocr INFO:         k : 50
[2026/03/07 10:38:48] ppocr INFO:         mode : large
[2026/03/07 10:38:48] ppocr INFO:         name : PFHeadLocal
[2026/03/07 10:38:48] ppocr INFO:     Neck : 
[2026/03/07 10:38:48] ppocr INFO:         intracl : True
[2026/03/07 10:38:48] ppocr INFO:         name : LKPAN
[2026/03/07 10:38:48] ppocr INFO:         out_chann

# PPOCR_V5_det Training

## Repo Clone & Installation

In [1]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import paddle

paddle.device.get_device()
paddle.set_device("gpu")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Place(gpu:0)

In [3]:
!git clone https://github.com/PaddlePaddle/PaddleOCR

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 338157, done.
remote: Counting objects: 100% (1086/1086), done.
remote: Compressing objects: 100% (296/296), done.
remote: Total 338157 (delta 918), reused 790 (delta 790), pack-reused 337071 (from 3)
Receiving objects: 100% (338157/338157), 1.79 GiB | 27.72 MiB/s, done.
Resolving deltas: 100% (267860/267860), done.


In [4]:
%cd PaddleOCR

!pip install -r requirements.txt

/content/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.0 MB/s eta 0:00:00


## Processing Data

In [6]:
from google.colab import drive
drive.mount('/content/Ibrah')

Mounted at /content/Ibrah


### Load Data: Unzipping from drive

In [ ]:
# -- unzip the root data zip --

root_zip_path = "/content/Ibrah/MyDrive/PNID_OCR_V2/Isometric Text Export.zip"
!unzip {root_zip_path.replace(" ", "\ ")} -d /content/annotated_data

In [ ]:
# unzip akhila and srvani annotatinos
akhila_annotation_path = "/content/annotated_data/Isometric Text Export/Akhila.zip"
srvani_annotation_path = "/content/annotated_data/Isometric Text Export/SRAVANI.zip"


!unzip {akhila_annotation_path.replace(" ", "\ ")} -d /content/akhila_data
!unzip {srvani_annotation_path.replace(" ", "\ ")} -d /content/srvani_data

### Utils

In [17]:
#====================================#
#   D A T A - L O A D - C L A S S    #
#====================================#

import os
import json
import math
import numpy as np
import cv2
import shutil
import json

def load_json(json_path):
  """load json from json path"""

  if os.path.exists(json_path) and json_path.lower().endswith(".json"):
    with open(json_path, "r") as f:
      jsons_list = json.load(f)
  else:
    raise ValueError(f"Invalid json path: {json_path}")

  return jsons_list



def read_image(image_path):
  """read image from the image path"""

  if image_path.lower().endswith(".jpg") or image_path.lower().endswith(".png"):
    image = cv2.imread(image_path)
  else:
    print("Invalid image path: {}".format(image_path))
    return None

  return image




def xywhrot_xyxyxyxy(label, percentages=True):
  """
  Converting Label Studio format into Oriented Bounding Box (OBB) format.

  Args:
    label (dict): Dictionary containing annotation information
    percentages (bool): if true it will convert percenatage values into pixels values

  Maths:
    label:
      x, y, width, height, rotation

      if percentages:
        x = x / 100 * original_width
        y = y / 100 * original_height
        w = width / 100 * original_width
        h = height / 100 * original_height

      rotation:
        rotation_ = math.radians(rotation)
        cos, sin = math.cos(rotation_), math.sin(rotation_)

        coords = [
          (x, y),
          (x + w * cos, y + w * sin),
          (x + w * cos - h * sin, y + w * sin + h * cos),
          (x - h * sin, y + h * cos),
        ]

  Returns:
    list of tuple or None: List of tuples containing the coordinates of the object in Yolo OBB
  """

  # unpick values
  original_width = label['original_width']
  original_height = label['original_height']
  x, y = label['x'], label['y']
  w, h = label['width'], label['height']
  rotation = label['rotation']


  if percentages:
    x = x / 100 * original_width
    y = y / 100 * original_height
    w = w / 100 * original_width
    h = h / 100 * original_height

  # rotation -> angles
  rotation = math.radians(rotation)
  cos, sin = math.cos(rotation), math.sin(rotation)

  # xywhr -> xyxyxyxy
  coords = [
      (x, y),
      (x + w * cos, y + w * sin),
      (x + w * cos -h * sin, y + w * sin + h * cos),
      (x - h * sin, y + h * cos)
  ]

  return np.array(coords, dtype=np.int32)





def xyxyxyxy_xywhr(xyxyxyxy, original_width, original_height):
  """
    Convert YOLO Oriented Bounding Box (OBB) format to Label Studio format.

    Args:
        xyxyxyxy (list): List of 8 float values representing the absolute pixel coordinates
                         of the OBB in the format [x1, y1, x2, y2, x3, y3, x4, y4]
        original_width (int): Original width of the image
        original_height (int): Original height of the image


    Return:
        dict: Dictionary containing the converted bounding box
              list: [x, y, width, height, rotation, original_width, original_height]
  """
  # Reshape the coordinates into a 4x2 matrix
  coords = np.array(xyxyxyxy, dtype=np.float64).reshape((4, 2))

  # Calculate the center of the bounding box
  center_x = np.mean(coords[:, 0])
  center_y = np.mean(coords[:, 1])

  # Calculate the width and height of the bounding box
  width = np.linalg.norm(coords[0] - coords[1])
  height = np.linalg.norm(coords[0] - coords[3])

  # Calculate the rotation angle
  dx = coords[1, 0] - coords[0, 0]
  dy = coords[1, 1] - coords[0, 1]
  r = np.degrees(np.arctan2(dy, dx))

  # Find the top-left corner (x, y)
  top_left_x = (
      center_x
      - (width / 2) * np.cos(np.radians(r))
      + (height / 2) * np.sin(np.radians(r))
  )
  top_left_y = (
      center_y
      - (width / 2) * np.sin(np.radians(r))
      - (height / 2) * np.cos(np.radians(r))
  )

  # Normalize the values
  x = (top_left_x / original_width) * 100
  y = (top_left_y / original_height) * 100
  width = (width / original_width) * 100
  height = (height / original_height) * 100

  # Create the dictionary for Label Studio
  return {
      "x": x,
      "y": y,
      "width": width,
      "height": height,
      "rotation": r,
      "original_width": original_width,
      "original_height": original_height,
  }


def put_text_on_polygon_simple(image, polygon, text,
                              font=cv2.FONT_HERSHEY_SIMPLEX,
                              font_scale=0.5,
                              thickness=1,
                              color=(0, 255, 0)):

    polygon = polygon.reshape(-1, 2).astype(int)

    # ---- 1. Get top-left point ----
    # smallest y → top, then smallest x → left
    top_left = sorted(polygon, key=lambda p: (p[1], p[0]))[0]

    x, y = top_left

    # ---- 2. Get text size ----
    (w, h), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    # ---- 3. Adjust position (so text is visible) ----
    y = max(h + 5, y)  # avoid going above image
    x = max(0, x)

    # ---- 4. Optional: background box for visibility ----
    cv2.rectangle(image,
                  (x, y - h - baseline),
                  (x + w, y + baseline),
                  (0, 0, 0))

    # ---- 5. Put horizontal text ----
    cv2.putText(image, text, (x, y),
                font, font_scale, color, thickness, cv2.LINE_AA)

    return image



def per_sample_object(image_path, bboxes, transcriptions):
  """It creates one row object per image

  Args:
    image_path (str): path to image
    bboxes (list): list of bounding boxes
    transcriptions (list): list of transcriptions


  Returns:
    list: list of per sample objects
    Example: [
                images/test.jpg,
                [
                    {
                        "transcription": "hello",
                        "points": [[0, 0], [0, 0], [0, 0], [0, 0]]
                    },
                    {
                        "transcription": "hello",
                        "points": [[0, 0], [0, 0], [0, 0], [0, 0]]
                    },
                    ...
                    ...
                    ...
                ]
            ]
  """

  objects = []
  for bbox, transcription in zip(bboxes, transcriptions):

    # xywhrot --> xyxyxyxy
    obb = xywhrot_xyxyxyxy(bbox)

    # append objects list
    objects.append({
        "transcription": transcription,
        "points": obb.tolist()
    })


  # return
  return [image_path, objects]




def copy_image(image_path_1, image_path_2):
  """copy image from image_path_1 to image_path_2"""

  if os.path.exists(image_path_1):
    os.makedirs(os.path.dirname(image_path_2), exist_ok=True)

    try:
      shutil.copy(image_path_1, image_path_2)
      return True
    except Exception as e:
      print("Error while copying image: {}".format(e))
      return False


def save_text_file(file_path, objects: list):
  """It saves the text file for the list of objects

  Args:
    file_path (str): path to text file
    objects (list): list of objects


  Returns:
    None
  """

  with open(file_path, "w") as f:
    for obj in objects:
      image_path = obj[0]
      objects_list = obj[1]

      f.write(image_path + "\t" + json.dumps(objects_list, ensure_ascii=False) + "\n")

### Dataset Visualization Script

In [ ]:
# -- unzip the root data zip --

root_zip_path = "/content/Ibrah/MyDrive/PNID_OCR_V2/Isometric Text Export.zip"
!unzip {root_zip_path.replace(" ", "\ ")} -d /content/annotated_data


# unzip akhila and srvani annotatinos
akhila_annotation_path = "/content/annotated_data/Isometric Text Export/Akhila.zip"
srvani_annotation_path = "/content/annotated_data/Isometric Text Export/SRAVANI.zip"


!unzip {akhila_annotation_path.replace(" ", "\ ")} -d /content/akhila_data
!unzip {srvani_annotation_path.replace(" ", "\ ")} -d /content/srvani_data

In [ ]:
# akhila data paths
'''
json_path = "/content/akhila_data/Akhila/project-3-at-2026-04-29-14-11-639dbd78.json"
root_images_path = "/content/akhila_data/Akhila/ocr_dataset"
save_results_path = "/content/saved_akhila"
'''

# srvani data paths
json_path = "/content/srvani_data/SRAVANI/project-4-at-2026-04-29-13-26-0b18cae9.json"
root_images_path = "/content/srvani_data/SRAVANI/ocr_dataset"
save_results_path = "/content/saved_srvani"


os.makedirs(save_results_path, exist_ok=True)

annotations = load_json(json_path)


# iterate annotations
for anno_idx, annotation in enumerate(annotations):

  # extract per image annotations and metadata
  image_name = annotation['ocr'].split("/")[-1]
  bboxes = annotation['bbox']
  transcriptions = annotation['transcription']


  # read image
  image_path = os.path.join(root_images_path, image_name)
  image = read_image(image_path)


  # loop through each bbox
  for bbox, transcription in zip(bboxes, transcriptions):
    obb = xywhrot_xyxyxyxy(bbox)

    cv2.polylines(image, [obb], True, (0, 255, 0), 2)
    image = put_text_on_polygon_simple(image, obb, transcription,
                        font=cv2.FONT_HERSHEY_SIMPLEX,
                        font_scale=0.5,
                        thickness=1,
                        color=(0, 0, 0))


  save image
  save_path = os.path.join(save_results_path, image_name)
  cv2.imwrite(save_path, image)
  print("Sample NO: {:04d} Processed and Saved Successfully to {}"
        .format(anno_idx, save_path))

### Main

In [26]:
if __name__=="__main__":


  # =========== DATASETS & PATHS CONFIGURATION =========== #
  CONFIG = {

      # akhila data paths
      'akhila': {
          'json_path' : "/content/akhila_data/Akhila/project-3-at-2026-04-29-14-11-639dbd78.json",
          'images_path' : "/content/akhila_data/Akhila/ocr_dataset"
      },

      # srvani data paths
      'srvani': {
          'json_path' : "/content/srvani_data/SRAVANI/project-4-at-2026-04-29-13-26-0b18cae9.json",
          'images_path' : "/content/srvani_data/SRAVANI/ocr_dataset"
      }
  }


  # det_model dataset structure
  DATASET_STRUCTURE = {
      "root" : "det_dataset/images",
      "images_path" : "images",
      "train_labels_path" : "det_dataset/train.txt",
      "val_labels_path" : "det_dataset/val.txt"
  }

  # create dataset structure
  os.makedirs(DATASET_STRUCTURE['root'], exist_ok=True)



  # =========== GLOBAL & LOCAL VARIABLES =========== #
  global_objects = []



  # =========== parase the CONFIG paths ============ #
  for annotater_name, annotation_configs in CONFIG.items():

    print("Processing Annotater {} annotated dataset".format(annotater_name))
    # paths to json and images
    json_path = annotation_configs['json_path']
    annotated_images_path = annotation_configs['images_path']


    # read json
    annotations = load_json(json_path)

    # parse annotations for each image
    # for anno_idx, annotation in enumerate(np.random.choice(annotations, size=5, replace=False)): # test_valid
    for anno_idx, annotation in enumerate(annotations):

      print("NO {} : Processing {}".format(anno_idx, annotation.get('ocr').split("/")[-1]))

      # extract iamge_name, bboxes, transcriptions
      image_name = annotation.get('ocr').split("/")[-1]
      bboxes = annotation.get('bbox')
      transcriptions = annotation.get('transcription')

      # image_path for the new dataset
      sample_path = os.path.join(DATASET_STRUCTURE['images_path'], image_name)
      image_path = os.path.join(DATASET_STRUCTURE['root'], image_name)

      # create object
      row_object = per_sample_object(
                        image_path=sample_path,
                        bboxes=bboxes,
                        transcriptions=transcriptions
                        )

      # copy image from to new dataset
      copy_image_flag = copy_image(
                        image_path_1=os.path.join(
                                            annotated_images_path,
                                            image_name),
                        image_path_2=image_path
                        )

      if not copy_image_flag:
        print("Failed to copy Image {} to {}".format(image_name, image_path))


      global_objects.append(row_object)


  # write annotation text file
  save_text_file(file_path=DATASET_STRUCTURE['train_labels_path'],
                 objects=global_objects
                 )

Processing Annotater akhila annotated dataset
NO 0 : Processing 1_page1__23b298f3-b16a-4b82-ba68-c30351d13464.jpg
NO 1 : Processing 1_page1__ad62b408-d37a-44d1-981e-4d45808f7d95.jpg
NO 2 : Processing 1_page1__cd75e8cd-e07c-43c0-b987-0a923af24abb.jpg
NO 3 : Processing 1_page1__09cb1245-91aa-4267-bfb8-6e125c710df9.jpg
NO 4 : Processing 2_page1__d86fa880-5012-4163-834b-2a7b421028fc.jpg
NO 5 : Processing 2_page1__c90d140d-3b76-40cb-94c7-a5c5b6ff8b74.jpg
NO 6 : Processing 2_page1__208ef36d-5d0c-4fce-95f7-d50291a0d1d1.jpg
NO 7 : Processing 251798840-JS1019-B-P03-1000-Part3_page-0001_page1__1f66b6e8-b6bf-4a27-8b7a-0e677f03962c.jpg
NO 8 : Processing 251798840-JS1019-B-P03-1000-Part3_page-0001_page1__f398e168-304b-4fa9-9839-b55c83b95a71.jpg
NO 9 : Processing 251798840-JS1019-B-P03-1000-Part3_page-0001_page1__3cabf37d-a36d-46e8-bc06-f1a25210efbd.jpg
NO 10 : Processing 251798840-JS1019-B-P03-1000-Part3_page-0001_page1__8a0f20b8-3f25-4ed4-a958-b9339cda0a42.jpg
NO 11 : Processing 251798840-JS1019-B

## Training Model

In [5]:
# 下载 PP-OCRv5_server_det 预训练模型
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams

--2026-05-01 16:11:57--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105414496 (101M) [application/octet-stream]
Saving to: ‘PP-OCRv5_server_det_pretrained.pdparams’

PP-OCRv5_server_det 100%[===================>] 100.53M  15.5MB/s    in 25s     

2026-05-01 16:12:24 (4.06 MB/s) - ‘PP-OCRv5_server_det_pretrained.pdparams’ saved [105414496/105414496]



In [ ]:
'''
Global:
  model_name: PP-OCRv5_server_det # To use static model for inference.
  debug: false
  use_gpu: true
  epoch_num: &epoch_num 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/PP-OCRv5_server_det
  save_epoch_step: 10
  eval_batch_step:
  - 0
  - 500
  cal_metric_during_train: false
  checkpoints:
  pretrained_model: https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PPHGNetV2_B4_ocr_det.pdparams
  save_inference_dir: null
  use_visualdl: false
  infer_img: doc/imgs_en/img_10.jpg
  save_res_path: ./checkpoints/det_db/predicts_db.txt
  distributed: true

Architecture:
  model_type: det
  algorithm: DB
  Transform: null
  Backbone:
    name: PPHGNetV2_B4
    det: True
  Neck:
    name: LKPAN
    out_channels: 256
    intracl: true
  Head:
    name: PFHeadLocal
    k: 50
    mode: "large"


Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005 #(8*8c)
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 1e-6

PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 2000
  unclip_ratio: 1.8

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./train_data/icdar2015/text_localization/
    label_file_list:
      - ./train_data/icdar2015/text_localization/train_icdar2015_label.txt
    ratio_list: [1.0]
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - CopyPaste: null
    - IaaAugment:
        augmenter_args:
        - type: Fliplr
          args:
            p: 0.5
        - type: Affine
          args:
            rotate:
            - -10
            - 10
        - type: Resize
          args:
            size:
            - 0.5
            - 2
    - EastRandomCropData:
        size:
        - 960
        - 960
        max_tries: 50
        keep_ratio: true
    - MakeBorderMap:
        shrink_ratio: 0.4
        thresh_min: 0.3
        thresh_max: 0.7
        total_epoch: *epoch_num
    - MakeShrinkMap:
        shrink_ratio: 0.4
        min_text_size: 6
        total_epoch: *epoch_num
    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - threshold_map
        - threshold_mask
        - shrink_map
        - shrink_mask
  loader:
    shuffle: true
    drop_last: false
    batch_size_per_card: 2
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./train_data/icdar2015/text_localization/
    label_file_list:
      - ./train_data/icdar2015/text_localization/test_icdar2015_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - DetResizeForTest:
        limit_side_len: 1792
        limit_type: max

    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - shape
        - polys
        - ignore_tags
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 2
profiler_options: null
'''

In [27]:
%cd /content/PaddleOCR

#单卡训练 (默认训练方式)
!python3 tools/train.py -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
    Train.dataset.data_dir=./det_dataset \
    Train.dataset.label_file_list='[./det_dataset/train.txt]' \
    Eval.dataset.data_dir=./det_dataset \
    Eval.dataset.label_file_list='[./det_dataset/val.txt]'


#多卡训练，通过--gpus参数指定卡号
# python3 -m paddle.distributed.launch --gpus '0,1,2,3' tools/train.py \
#     -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
#     -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
#     Train.dataset.data_dir=./ocr_det_dataset_examples \
#     Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
#     Eval.dataset.data_dir=./ocr_det_dataset_examples \
#     Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'

/content/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/05/01 16:52:09] ppocr INFO: Architecture : 
[2026/05/01 16:52:09] ppocr INFO:     Backbone : 
[2026/05/01 16:52:09] ppocr INFO:         det : True
[2026/05/01 16:52:09] ppocr INFO:         name : PPHGNetV2_B4
[2026/05/01 16:52:09] ppocr INFO:     Head : 
[2026/05/01 16:52:09] ppocr INFO:         k : 50
[2026/05/01 16:52:09] ppocr INFO:         mode : large
[2026/05/01 16:52:09] ppocr INFO:         name : PFHeadLocal
[2026/05/01 16:52:09] ppocr INFO:     Neck : 
[2026/05/01 16:52:09] ppocr INFO:         intracl : True
[2026/05/01 16:52:09] ppocr INFO:         name : LKPAN
[2026/05/01 16:52:09] ppocr INFO

In [29]:
!python3 tools/export_model.py \
    -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./output/PP-OCRv5_server_det/best_model/model.pdparams \
       Global.save_inference_dir=./inference/det_model

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
W0501 18:34:25.148455 47418 gpu_resources.cc:116] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 13.0
[2026/05/01 18:34:25] ppocr INFO: load pretrain successful from ./output/PP-OCRv5_server_det/best_model/model
[2026/05/01 18:34:25] ppocr INFO: Export inference config file to ./inference/det_model/inference.yml
Skipping import of the encryption module
W0501 18:34:28.628171 47418 eager_utils.cc:3584] Paddle static graph(PIR) not support input out tensor for now!!!!!
[2026/05/01 18:34:30] ppocr INFO: inference model is saved to ./inference/det_model/inference


## Testing Model

In [40]:
!python3 tools/infer/predict_det.py \
    --det_model_dir=./inference/det_model \
    --image_dir=./det_dataset/images/IMG-20260305-WA0016.jpg \
    --use_gpu=True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/05/01 18:45:22] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/05/01 18:45:24] ppocr INFO: IMG-20260305-WA0016.jpg	[[[38.0, 306.0], [205.0, 306.0], [205.0, 320.0], [38.0, 320.0]], [[435.0, 296.0], [573.0, 296.0], [573.0, 309.0], [435.0, 309.0]], [[131.0, 260.0], [149.0, 260.0], [149.0, 277.0], [131.0, 277.0]], [[511.0, 242.0], [535.0, 257.0], [522.0, 276.0], [498.0, 261.0]], [[142.0, 261.0], [200.0, 229.0], [213.0, 249.0], [154.0, 281.0]], [[566.0, 220.0], [591.0, 232.0], [580.0, 252.0], [555.0, 240.0]], [[63.0, 209.0], [116.0, 240.0], [103.0, 261.0], [50.0, 230.0]], [[409.0, 194.0], [442.0, 199.0], [439.0, 222.0], [406.0, 217.0]],

In [ ]:
from paddleocr import TextDetection
model = TextDetection(model_name="/content/PaddleOCR/output/PP-OCRv5_server_det/best_model/model")
output = model.predict("/content/PaddleOCR/det_dataset/images/251798840-JS1019-B-P03-1000-Part3_page-0004_page1__1f97773c-004a-47e7-a5f7-d8b7d944a0eb", batch_size=1)
for res in output:
    res.print()
    res.save_to_img(save_path="./output_1/")
    res.save_to_json(save_path="./output_1/res.json")